# Junction Decode Analysis

Analyze what the neural decoder represents at every 3-way junction in the maze.
For each junction encounter: does the decode anticipate the left or right branch,
and which way did the rat actually go?

In [ ]:
import matplotlib.pyplot as plt
import datajoint as dj
import numpy as np
import pandas as pd

from hexmaze import (
    plot_hex_maze,
    get_all_choice_points,
    get_critical_choice_points,
    get_choice_direction,
    get_junction_left_right_map,
    classify_exit_direction,
    get_hex_centroids,
    maze_to_graph,
)

import spyglass.common as sgc
from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock, HexMazeChoice, HexCentroids
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeJunctionDecode,
    HexMazeDecodedPositionHexAnnotated,
)

# Select session
key = {"nwb_file_name": "Toby20250316_.nwb", "epoch": 5}
display(HexMazeBlock() & key)

## Validate cross-product sign convention

Confirm that the geometric left/right classification at the critical choice point
matches the existing `get_choice_direction(start_port, end_port)` for all trials.

In [ ]:
# Get standard centroids and trial data
centroids = get_hex_centroids()

# Fetch all trials with choice direction for this session
trials_with_choice = (HexMazeBlock().Trial() * HexMazeChoice() & key).fetch(as_dict=True)

matches = 0
mismatches = 0

for trial in trials_with_choice:
    maze = (HexMazeBlock() & {
        "nwb_file_name": trial["nwb_file_name"],
        "block": trial["block"],
        "epoch": trial["epoch"],
    }).fetch1("config_id")

    start_port = trial["start_port"]
    if start_port == "None":
        continue

    # Get critical choice points for this start port
    crit_cps = get_critical_choice_points(maze, start_port)
    if not crit_cps:
        continue

    # Get the junction left/right map
    lr_map = get_junction_left_right_map(maze, centroids)

    # For the critical choice point, we need to know the entry direction.
    # The entry comes from the start section. Use the graph to find which
    # neighbor of the choice point is in the start section.
    from hexmaze import divide_into_thirds
    port_map = {"A": 1, "B": 2, "C": 3}
    thirds = divide_into_thirds(maze)
    start_section_hexes = set(thirds[port_map[start_port] - 1])
    graph = maze_to_graph(maze)

    for cp in crit_cps:
        # Find which neighbor of the choice point is in the start section
        neighbors = list(graph.neighbors(cp))
        entry_candidates = [n for n in neighbors if n in start_section_hexes]
        if not entry_candidates:
            continue
        entry_hex = entry_candidates[0]

        if (cp, entry_hex) not in lr_map:
            continue

        lr = lr_map[(cp, entry_hex)]

        # The end_port tells us which exit the rat took
        end_port = trial["end_port"]
        end_section_hexes = set(thirds[port_map[end_port] - 1])

        # Which exit leads to the chosen port's section?
        left_in_chosen = lr["left"] in end_section_hexes
        right_in_chosen = lr["right"] in end_section_hexes

        if left_in_chosen:
            geometric_direction = "left"
        elif right_in_chosen:
            geometric_direction = "right"
        else:
            # Exit neighbors might not be directly in the end section
            # (could be in the choice point section itself)
            continue

        expected = trial["choice_direction"]
        if geometric_direction == expected:
            matches += 1
        else:
            mismatches += 1

print(f"Matches: {matches}, Mismatches: {mismatches}")
if mismatches == 0 and matches > 0:
    print("Cross-product sign convention is CORRECT")
elif mismatches > 0 and matches == 0:
    print("WARNING: Sign convention is INVERTED — need to flip left/right")
else:
    print(f"Mixed results — investigate ({matches}/{matches+mismatches} correct)")

## Fetch junction decode data

In [ ]:
# Populate the table if not already done
# HexMazeJunctionDecode.populate(key)

df = (HexMazeJunctionDecode & key).fetch1_dataframe()

# Filter out timepoints outside trials
df = df[df["hex"] != -100]

# Summary of junctions found
junc_df = df[df["nearest_junction"] != -100]
print(f"Total timepoints: {len(df):,}")
print(f"Timepoints near a junction: {len(junc_df):,} ({100*len(junc_df)/len(df):.1f}%)")
print(f"Unique junctions visited: {junc_df['nearest_junction'].nunique()}")
print()

# Count junction encounters (at the junction itself, hexes_from_junction == 0)
at_junc = df[df["hexes_from_junction"] == 0]
encounters = at_junc.groupby(["epoch_trial_num", "nearest_junction"]).ngroups
print(f"Total junction encounters: {encounters}")

## Decode prediction accuracy at junctions

When the rat is at a junction, does the decode point toward the side the rat will choose?

In [ ]:
# Filter to high-confidence timepoints at junctions
# Each row is one decoded timepoint while the rat is on the junction hex.
# We require low spatial covariance (tight posterior), sufficient speed
# (not stationary), and valid left/right labels for both rat and decode.
filtered = df[
    (df["spatial_cov"] < 500)
    & (df["speed"] > 5)
    & (df["hexes_from_junction"] == 0)
    & (df["rat_junction_direction"].isin(["left", "right"]))
    & (df["decode_junction_direction"].isin(["left", "right"]))
    & (df["epoch_trial_num"] > 1)
]

# For each timepoint, check whether the decoded direction matches the
# direction the rat actually chose. The decode can fluctuate between
# left and right within a single pass through the junction, so this is
# not a binary per-trial measure — it's a per-timepoint match.
filtered = filtered.copy()
filtered["decode_matches"] = filtered["decode_junction_direction"] == filtered["rat_junction_direction"]

# Build a per-block map of critical choice points.
# Different blocks can have different maze configs (e.g., barrier changes),
# so which junctions are "critical" depends on the block.
block_to_maze = {}
block_critical_cps = {}
for block_row in (HexMazeBlock() & key).fetch(as_dict=True):
    maze = block_row["config_id"]
    block_to_maze[block_row["block"]] = maze
    # get_critical_choice_points without a start_port returns all CPs
    # that are critical for ANY start port in this maze
    block_critical_cps[block_row["block"]] = get_critical_choice_points(maze)

# Label each timepoint as critical CP based on its own block's maze config
filtered["is_critical_cp"] = filtered.apply(
    lambda row: row["nearest_junction"] in block_critical_cps.get(row["block"], set()),
    axis=1,
)

# Compute per-encounter accuracy: for each trial x junction, average the
# timepoint-level decode_matches to get the fraction of timepoints where
# the decode pointed toward the rat's eventual choice. This gives one
# continuous value (0–1) per encounter, NOT a binary correct/incorrect.
# n below = number of such encounters.
trial_acc = filtered.groupby(
    ["epoch_trial_num", "nearest_junction", "is_critical_cp"]
)["decode_matches"].mean().reset_index(name="accuracy")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall accuracy: critical vs non-critical
for label, group in trial_acc.groupby("is_critical_cp"):
    name = "Critical CP" if label else "Other junctions"
    mean_acc = group["accuracy"].mean()
    sem_acc = group["accuracy"].sem()
    n = len(group)
    axes[0].bar(name, mean_acc, yerr=sem_acc, capsize=5)
    axes[0].annotate(f"n={n}", (name, mean_acc), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=9)

axes[0].axhline(0.5, color="k", linestyle="--", alpha=0.5, label="Chance")
axes[0].set_ylabel("Fraction decode matches rat direction")
axes[0].set_title("Decode accuracy at junctions")
axes[0].legend()

# Per-junction accuracy, with critical CPs highlighted.
# Note: a junction hex could be critical in one block but not another.
# Here we use the majority label across encounters for coloring, but the
# per-encounter is_critical_cp is correct per-block above.
junc_acc = trial_acc.groupby(["nearest_junction", "is_critical_cp"])["accuracy"].agg(
    ["mean", "sem", "count"]
).reset_index()
junc_acc = junc_acc.sort_values("mean", ascending=False).reset_index(drop=True)

# Color bars by whether the junction is a critical choice point
colors = ["tab:orange" if row["is_critical_cp"] else "tab:blue" for _, row in junc_acc.iterrows()]
bars = axes[1].bar(range(len(junc_acc)), junc_acc["mean"], yerr=junc_acc["sem"],
                   capsize=3, color=colors)

# Add n (number of encounters) above each bar
for i, row in junc_acc.iterrows():
    axes[1].annotate(f"n={int(row['count'])}", (i, row["mean"]),
                     textcoords="offset points", xytext=(0, 10),
                     ha="center", fontsize=7)

axes[1].set_xticks(range(len(junc_acc)))
axes[1].set_xticklabels(junc_acc["nearest_junction"], rotation=45)
axes[1].axhline(0.5, color="k", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Junction hex")
axes[1].set_ylabel("Fraction decode matches rat direction")
axes[1].set_title("Decode accuracy by junction")

# Add legend for critical CP highlighting
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(facecolor="tab:orange", label="Critical CP"),
    Patch(facecolor="tab:blue", label="Other junction"),
], loc="upper right")

plt.tight_layout()

### Per-encounter breakdown

Each dot is one trial's pass through a junction. The y-axis shows the fraction
of decoded timepoints during that encounter that matched the rat's eventual
choice. A value of 1.0 means the decode consistently pointed the correct way;
0.0 means it consistently pointed the wrong way; values near 0.5 indicate
the decode was split between left and right.

Encounters are grouped by junction AND entry direction, since the same junction
hex has different left/right exit assignments depending on which neighbor the
rat entered from.

In [ ]:
# ---- Self-contained: configure which distance from junction to analyze ----
# Set to 0 for "at the junction", -1 for "one hex before arrival", etc.
HEXES_FROM_JUNCTION = -1

def _most_common_frac(directions):
    """Fraction of encounters where the rat chose the most common direction."""
    counts = directions.value_counts()
    return counts.iloc[0] / len(directions)

def _most_common_label(directions):
    """Label string like '8L/2R' with the most common direction first."""
    n_left = (directions == "left").sum()
    n_right = (directions == "right").sum()
    if n_left >= n_right:
        return f"{n_left}L/{n_right}R"
    else:
        return f"{n_right}R/{n_left}L"

# Start from ALL timepoints at this distance from a junction (before quality filters)
# so we can count how many encounters had no usable decode output.
_all_at_dist = df[
    (df["hexes_from_junction"] == HEXES_FROM_JUNCTION)
    & (df["rat_junction_direction"].isin(["left", "right"]))
    & (df["epoch_trial_num"] > 1)
]

# Per-encounter rat direction for ALL encounters (before quality filtering)
all_encounter_stats = _all_at_dist.groupby(
    ["epoch_trial_num", "nearest_junction", "junction_entry_hex"]
).agg(
    rat_direction=("rat_junction_direction", "first"),
).reset_index()

# Count all encounters per (junction, entry_hex)
total_encounter_counts = all_encounter_stats.groupby(
    ["nearest_junction", "junction_entry_hex"]
).size().reset_index(name="n_total")

# Summarize choice bias across ALL encounters (before filtering)
all_choice_summary = all_encounter_stats.groupby(
    ["nearest_junction", "junction_entry_hex"]
).agg(
    frac_most_common_all=("rat_direction", _most_common_frac),
    choice_label_all=("rat_direction", _most_common_label),
).reset_index()

# Now apply quality filters
_filtered = df[
    (df["spatial_cov"] < 500)
    & (df["speed"] > 5)
    & (df["hexes_from_junction"] == HEXES_FROM_JUNCTION)
    & (df["rat_junction_direction"].isin(["left", "right"]))
    & (df["decode_junction_direction"].isin(["left", "right"]))
    & (df["epoch_trial_num"] > 1)
].copy()

_filtered["decode_matches"] = _filtered["decode_junction_direction"] == _filtered["rat_junction_direction"]

# Build per-block critical CP map (same logic as prediction cell)
_block_critical_cps = {}
for block_row in (HexMazeBlock() & key).fetch(as_dict=True):
    maze = block_row["config_id"]
    _block_critical_cps[block_row["block"]] = get_critical_choice_points(maze)

_filtered["is_critical_cp"] = _filtered.apply(
    lambda row: row["nearest_junction"] in _block_critical_cps.get(row["block"], set()),
    axis=1,
)

# Per-encounter stats: for each trial pass through a (junction, entry_hex),
# get the decode accuracy AND the rat's actual choice direction.
# We take the rat direction from the first timepoint (it's constant within
# an encounter — the rat only goes one way).
encounter_stats = _filtered.groupby(
    ["epoch_trial_num", "nearest_junction", "junction_entry_hex", "is_critical_cp"]
).agg(
    accuracy=("decode_matches", "mean"),
    rat_direction=("rat_junction_direction", "first"),
    n_timepoints=("decode_matches", "count"),
).reset_index()

# Count filtered encounters per (junction, entry_hex)
filtered_encounter_counts = encounter_stats.groupby(
    ["nearest_junction", "junction_entry_hex"]
).size().reset_index(name="n_filtered")

# Merge to get n_dropped = encounters that had zero usable timepoints
encounter_yield = total_encounter_counts.merge(
    filtered_encounter_counts, on=["nearest_junction", "junction_entry_hex"], how="left"
)
encounter_yield["n_filtered"] = encounter_yield["n_filtered"].fillna(0).astype(int)
encounter_yield["n_dropped"] = encounter_yield["n_total"] - encounter_yield["n_filtered"]

entry_summary = encounter_stats.groupby(
    ["nearest_junction", "junction_entry_hex", "is_critical_cp"]
).agg(
    mean_accuracy=("accuracy", "mean"),
    sem_accuracy=("accuracy", "sem"),
    n_encounters=("accuracy", "count"),
    # Choice bias from usable-decode encounters only
    frac_most_common=("rat_direction", _most_common_frac),
    choice_label=("rat_direction", _most_common_label),
).reset_index()

# Merge in encounter yield and ALL-encounter choice bias
entry_summary = entry_summary.merge(
    encounter_yield, on=["nearest_junction", "junction_entry_hex"], how="left"
)
entry_summary = entry_summary.merge(
    all_choice_summary, on=["nearest_junction", "junction_entry_hex"], how="left"
)
entry_summary = entry_summary.sort_values("mean_accuracy", ascending=False).reset_index(drop=True)

# Build x-axis labels like "junction←entry"
entry_summary["label"] = (
    entry_summary["nearest_junction"].astype(int).astype(str)
    + "←"
    + entry_summary["junction_entry_hex"].astype(int).astype(str)
)

label_to_x = {row["label"]: i for i, row in entry_summary.iterrows()}
pair_to_crit = dict(zip(
    zip(entry_summary["nearest_junction"], entry_summary["junction_entry_hex"]),
    entry_summary["is_critical_cp"],
))

fig, axes = plt.subplots(3, 1, figsize=(max(14, len(entry_summary) * 0.8), 13),
                         sharex=True, gridspec_kw={"height_ratios": [3, 1.2, 1]})

# --- Top panel: decode accuracy with per-encounter dots ---
ax = axes[0]
for _, row in encounter_stats.iterrows():
    pair = (row["nearest_junction"], row["junction_entry_hex"])
    label = f"{int(row['nearest_junction'])}←{int(row['junction_entry_hex'])}"
    if label not in label_to_x:
        continue
    x = label_to_x[label]
    is_crit = pair_to_crit.get(pair, False)
    color = "tab:orange" if is_crit else "tab:blue"
    jitter = np.random.uniform(-0.2, 0.2)
    ax.plot(x + jitter, row["accuracy"], "o", color=color, alpha=0.5, markersize=5)

# Overlay mean as black horizontal bars, annotate n
for i, row in entry_summary.iterrows():
    ax.plot([i - 0.3, i + 0.3], [row["mean_accuracy"], row["mean_accuracy"]], "k-", lw=2)
    ax.annotate(f"n={int(row['n_encounters'])}", (i, row["mean_accuracy"]),
                textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=7)

ax.axhline(0.5, color="k", linestyle="--", alpha=0.5, label="Chance")
ax.set_ylabel("Fraction of timepoints\ndecode matches rat direction")
dist_label = "at junction" if HEXES_FROM_JUNCTION == 0 else f"{abs(HEXES_FROM_JUNCTION)} hex(es) before junction"
ax.set_title(f"Per-encounter decode accuracy by junction and entry direction ({dist_label})")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
ax.legend(handles=[
    Patch(facecolor="tab:orange", alpha=0.5, label="Critical CP"),
    Patch(facecolor="tab:blue", alpha=0.5, label="Other junction"),
    Line2D([0], [0], color="k", lw=2, label="Mean"),
], loc="upper right")

# --- Middle panel: rat choice bias — all encounters vs usable-decode only ---
# Side-by-side bars: left = all encounters, right = usable encounters.
# If filtering preferentially drops one choice direction, these will diverge.
ax2 = axes[1]
bar_width = 0.3
for i, row in entry_summary.iterrows():
    is_crit = row["is_critical_cp"]
    color = "tab:orange" if is_crit else "tab:blue"
    # All encounters (lighter, left)
    ax2.bar(i - bar_width / 2, row["frac_most_common_all"], color=color,
            alpha=0.3, width=bar_width, edgecolor=color)
    ax2.annotate(row["choice_label_all"], (i - bar_width / 2, row["frac_most_common_all"]),
                 textcoords="offset points", xytext=(0, 5),
                 ha="center", fontsize=6, color="gray")
    # Usable encounters (solid, right)
    ax2.bar(i + bar_width / 2, row["frac_most_common"], color=color,
            alpha=0.7, width=bar_width)
    ax2.annotate(row["choice_label"], (i + bar_width / 2, row["frac_most_common"]),
                 textcoords="offset points", xytext=(0, 5),
                 ha="center", fontsize=6)

ax2.axhline(0.5, color="k", linestyle="--", alpha=0.5)
ax2.set_ylim(0.4, 1.05)
ax2.set_ylabel("Fraction chose\nmost common direction")
ax2.legend(handles=[
    Patch(facecolor="tab:blue", alpha=0.3, edgecolor="tab:blue", label="All encounters"),
    Patch(facecolor="tab:blue", alpha=0.7, label="Usable decode only"),
], loc="upper right", fontsize=8)

# --- Bottom panel: encounter yield (how many encounters had usable decode) ---
# Shows total encounters as full bar, with the dropped portion in gray.
# High drop rates mean the decode was often low-confidence or the rat was
# stationary at this junction — results based on few survivors may be noisy.
ax3 = axes[2]
for i, row in entry_summary.iterrows():
    is_crit = row["is_critical_cp"]
    color = "tab:orange" if is_crit else "tab:blue"
    # Total encounters (lighter, behind)
    ax3.bar(i, row["n_total"], color="lightgray", width=0.6)
    # Usable encounters (colored, in front)
    ax3.bar(i, row["n_filtered"], color=color, alpha=0.7, width=0.6)
    ax3.annotate(f"{int(row['n_filtered'])}/{int(row['n_total'])}",
                 (i, row["n_total"]),
                 textcoords="offset points", xytext=(0, 5),
                 ha="center", fontsize=7)

ax3.set_ylabel("Encounters\n(usable / total)")
ax3.set_xlabel("Junction ← Entry hex")
ax3.set_xticks(range(len(entry_summary)))
ax3.set_xticklabels(entry_summary["label"], rotation=45, ha="right")

from matplotlib.patches import Patch as _Patch
ax3.legend(handles=[
    _Patch(facecolor="lightgray", label="No usable decode"),
    _Patch(facecolor="tab:blue", alpha=0.7, label="Usable encounters"),
], loc="upper right", fontsize=8)

plt.tight_layout()

In [ ]:
# Same breakdown but weighted by time: encounters with more timepoints
# contribute proportionally more to the mean. Instead of averaging
# per-encounter accuracies (which gives equal weight to a 5-timepoint
# pass and a 50-timepoint pass), this pools all timepoints for each
# (junction, entry_hex) pair and computes the overall fraction correct.

# Count correct and total timepoints per encounter
encounter_counts = filtered.groupby(
    ["epoch_trial_num", "nearest_junction", "junction_entry_hex", "is_critical_cp"]
)["decode_matches"].agg(["sum", "count"]).reset_index()
encounter_counts.columns = [
    "epoch_trial_num", "nearest_junction", "junction_entry_hex",
    "is_critical_cp", "n_correct", "n_total",
]

# Pool timepoints across encounters for each (junction, entry_hex)
pooled = encounter_counts.groupby(
    ["nearest_junction", "junction_entry_hex", "is_critical_cp"]
)[["n_correct", "n_total"]].sum().reset_index()
pooled["accuracy"] = pooled["n_correct"] / pooled["n_total"]

# For SEM: use binomial SE = sqrt(p*(1-p)/n) since each timepoint is a
# binary match/mismatch and we're pooling across all of them
pooled["sem"] = np.sqrt(
    pooled["accuracy"] * (1 - pooled["accuracy"]) / pooled["n_total"]
)
pooled = pooled.sort_values("accuracy", ascending=False).reset_index(drop=True)

# Build labels and mappings (same format as unweighted plot)
pooled["label"] = (
    pooled["nearest_junction"].astype(int).astype(str)
    + "←"
    + pooled["junction_entry_hex"].astype(int).astype(str)
)
label_to_x = {row["label"]: i for i, row in pooled.iterrows()}
pair_to_crit = dict(zip(
    zip(pooled["nearest_junction"], pooled["junction_entry_hex"]),
    pooled["is_critical_cp"],
))

fig, ax = plt.subplots(figsize=(max(14, len(pooled) * 0.8), 6))

# Plot per-encounter accuracies as jittered dots (same as before, for context)
for _, row in encounter_counts.iterrows():
    label = f"{int(row['nearest_junction'])}←{int(row['junction_entry_hex'])}"
    if label not in label_to_x:
        continue
    x = label_to_x[label]
    pair = (row["nearest_junction"], row["junction_entry_hex"])
    is_crit = pair_to_crit.get(pair, False)
    color = "tab:orange" if is_crit else "tab:blue"
    # Scale dot size by number of timepoints in that encounter
    size = np.clip(row["n_total"] / 2, 3, 20)
    jitter = np.random.uniform(-0.2, 0.2)
    enc_acc = row["n_correct"] / row["n_total"]
    ax.plot(x + jitter, enc_acc, "o", color=color, alpha=0.4, markersize=size)

# Overlay pooled (time-weighted) mean as black horizontal bars
for i, row in pooled.iterrows():
    ax.plot([i - 0.3, i + 0.3], [row["accuracy"], row["accuracy"]], "k-", lw=2)
    ax.annotate(f"n={int(row['n_total'])}tp", (i, row["accuracy"]),
                textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=7)

ax.axhline(0.5, color="k", linestyle="--", alpha=0.5, label="Chance")
ax.set_xticks(range(len(pooled)))
ax.set_xticklabels(pooled["label"], rotation=45, ha="right")
ax.set_xlabel("Junction ← Entry hex")
ax.set_ylabel("Fraction of timepoints decode matches rat direction")
ax.set_title("Per-encounter decode accuracy by junction and entry direction\n(time-weighted: mean pools all timepoints, dot size ∝ encounter duration)")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
ax.legend(handles=[
    Patch(facecolor="tab:orange", alpha=0.5, label="Critical CP"),
    Patch(facecolor="tab:blue", alpha=0.5, label="Other junction"),
    Line2D([0], [0], color="k", lw=2, label="Pooled mean"),
])

plt.tight_layout()

## Decode anticipation

Does the decode predict the rat's turn *before* it arrives at the junction?
Plot decode accuracy as a function of distance from the junction.

In [ ]:
filtered = df[
    (df["spatial_cov"] < 500)
    & (df["speed"] > 5)
    & (df["nearest_junction"] != -100)
    & (df["rat_junction_direction"].isin(["left", "right"]))
    & (df["decode_junction_direction"].isin(["left", "right"]))
    & (df["epoch_trial_num"] > 1)
].copy()

filtered["decode_matches"] = filtered["decode_junction_direction"] == filtered["rat_junction_direction"]

# Per-trial accuracy at each distance from junction
trial_by_dist = filtered.groupby(
    ["epoch_trial_num", "hexes_from_junction"]
)["decode_matches"].mean().reset_index(name="accuracy")

stats = trial_by_dist.groupby("hexes_from_junction")["accuracy"].agg(["mean", "sem", "count"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(stats.index, stats["mean"], yerr=stats["sem"], marker="o", capsize=3)
ax.axhline(0.5, color="k", linestyle="--", alpha=0.5, label="Chance")
ax.axvline(0, color="gray", linestyle=":", alpha=0.5, label="At junction")
ax.set_xlabel("Hexes from junction (negative = approaching)")
ax.set_ylabel("Fraction decode matches rat direction")
ax.set_title("Decode anticipation around junctions")
ax.legend()

# Annotate with trial counts
for x, row in stats.iterrows():
    ax.annotate(f'n={int(row["count"])}', (x, row["mean"]),
               textcoords="offset points", xytext=(0, 10), ha="center", fontsize=7)

plt.tight_layout()

## Dead-end junctions

At junctions where one exit leads to a dead end, the "correct" choice is more
deterministic. Does the decode show stronger prediction here?

In [ ]:
from hexmaze import classify_maze_hexes

# For each block's maze, identify which junction exits lead to dead ends.
# This must be done per-block since different blocks can have different mazes.
block_dead_end_junctions = {}

for block_row in (HexMazeBlock() & key).fetch(as_dict=True):
    maze = block_row["config_id"]
    classification = classify_maze_hexes(maze)
    dead_end_hexes = classification.get("dead_end_hexes", set())
    junctions = get_all_choice_points(maze)
    graph = maze_to_graph(maze)

    dead_end_set = set()
    for j in junctions:
        neighbors = set(graph.neighbors(j))
        # A junction has a dead-end exit if any neighbor is a dead-end hex
        if neighbors & dead_end_hexes:
            dead_end_set.add(j)
    block_dead_end_junctions[block_row["block"]] = dead_end_set

print("Junctions with a dead-end exit per block:")
for b, de in block_dead_end_junctions.items():
    print(f"  Block {b}: {sorted(de)}")

# Compare decode accuracy at dead-end junctions vs non-dead-end junctions
filtered = df[
    (df["spatial_cov"] < 500)
    & (df["speed"] > 5)
    & (df["hexes_from_junction"] == 0)
    & (df["rat_junction_direction"].isin(["left", "right"]))
    & (df["decode_junction_direction"].isin(["left", "right"]))
    & (df["epoch_trial_num"] > 1)
].copy()

filtered["decode_matches"] = filtered["decode_junction_direction"] == filtered["rat_junction_direction"]

# Label per-timepoint using the correct block's dead-end junctions
filtered["is_dead_end_junction"] = filtered.apply(
    lambda row: row["nearest_junction"] in block_dead_end_junctions.get(row["block"], set()),
    axis=1,
)

trial_acc = filtered.groupby(
    ["epoch_trial_num", "is_dead_end_junction"]
)["decode_matches"].mean().reset_index(name="accuracy")

fig, ax = plt.subplots(figsize=(6, 5))
for label, group in trial_acc.groupby("is_dead_end_junction"):
    name = "Dead-end junction" if label else "Non-dead-end junction"
    mean_acc = group["accuracy"].mean()
    sem_acc = group["accuracy"].sem()
    ax.bar(name, mean_acc, yerr=sem_acc, capsize=5)
    ax.annotate(f"n={len(group)}", (name, mean_acc), textcoords="offset points",
               xytext=(0, 10), ha="center", fontsize=9)

ax.axhline(0.5, color="k", linestyle="--", alpha=0.5)
ax.set_ylabel("Fraction decode matches rat direction")
ax.set_title("Decode accuracy: dead-end vs non-dead-end junctions")
plt.tight_layout()

## Maze visualization

Plot the maze with junction left/right exits labeled.

In [ ]:
# Plot each block's maze with junctions labeled and critical CPs highlighted
centroids = get_hex_centroids()
blocks = (HexMazeBlock() & key).fetch(as_dict=True, order_by="block")

fig, axes = plt.subplots(1, len(blocks), figsize=(12 * len(blocks), 10))
if len(blocks) == 1:
    axes = [axes]

for ax, block_row in zip(axes, blocks):
    maze = block_row["config_id"]
    block_num = block_row["block"]

    lr_map = get_junction_left_right_map(maze, centroids)
    junctions = get_all_choice_points(maze)
    critical_cps = get_critical_choice_points(maze)
    graph = maze_to_graph(maze)

    plot_hex_maze(maze, centroids=centroids, ax=ax)

    # For each junction, pick the first entry direction and draw left/right arrows
    for j in junctions:
        neighbors = sorted(graph.neighbors(j))
        entry = neighbors[0]
        if (j, entry) not in lr_map:
            continue
        lr = lr_map[(j, entry)]
        jx, jy = centroids[j]

        # Draw arrows to left (blue) and right (red) exits
        for direction, color in [("left", "blue"), ("right", "red")]:
            ex, ey = centroids[lr[direction]]
            dx, dy = (ex - jx) * 0.4, (ey - jy) * 0.4
            ax.annotate("", xy=(jx + dx, jy + dy), xytext=(jx, jy),
                        arrowprops=dict(arrowstyle="->", color=color, lw=2))

        # Label the junction — orange bold for critical CPs, black for others
        is_crit = j in critical_cps
        ax.text(jx, jy, str(j), ha="center", va="center", fontsize=8,
                fontweight="bold",
                color="tab:orange" if is_crit else "black",
                bbox=dict(facecolor="white", edgecolor="tab:orange" if is_crit else "none",
                          boxstyle="round,pad=0.2", alpha=0.8) if is_crit else None)

    ax.set_title(f"Block {block_num} — Maze {maze}\n"
                 f"Left (blue) / Right (red) exits  |  "
                 f"Critical CPs in orange\n"
                 f"(entry from first neighbor)")

plt.tight_layout()